# Generate synthetic Latin stone inscriptions

This notebook runs **`make_synth.py`** to (re)create the synthetic training set in
`data/synth/` — the exact folder the training notebook (`htr_cluster_demo.ipynb`)
trains on. Run this to refresh that data or make your own; then run the training
notebook to use it.

The output carries pixel-exact PAGE-XML ground truth. No photographs are involved, so
it is freely licensable and needs no manual annotation.

**Pipeline per image:** text → layout → Capitalis glyphs → weathered carved relief
(erosion, crossbar loss, directional lighting, cracks, pits) on real or procedural
stone → camera (affine warp, vignette, exposure/B&W fade, blur, JPEG). A GPU is **not**
needed (only Pillow + numpy).

Middots (·) between words are **on by default**, matching the real inscriptions in
`data/real/` (which carry an interpunct where the stone shows one). Pass `--no-dots`
for pure *scriptio continua*.

## 1. Setup
Two dependencies. The 5 OFL fonts in `fonts/` and the stone textures in `stone_images/`
are already bundled, so this works offline on the cluster.

In [ ]:
import sys
!{sys.executable} -m pip install --quiet pillow numpy
print("ready")

## 2. Generate the training set
This clears `data/synth/` and regenerates **500** images there (about 2–3 minutes on a
CPU). Texts are drawn from `inscription_texts.txt` (50,000 real EDH transcriptions,
CC BY-SA 4.0); each image picks one of the 5 fonts and a stone texture at random
(seeded, so it is reproducible). Lower `--n` for a quick look.

In [ ]:
import shutil
shutil.rmtree("data/synth", ignore_errors=True)
!{sys.executable} make_synth.py --n 500 --outdir data/synth --seed 42

## 3. Look at what we made

In [ ]:
from pathlib import Path
from PIL import Image
from IPython.display import display

imgs = sorted(Path("data/synth/images").glob("*.jpg"))[:6]
W, H = 320, 240
grid = Image.new("RGB", (W * 3, H * 2), "white")
for i, p in enumerate(imgs):
    grid.paste(Image.open(p).resize((W, H)), ((i % 3) * W, (i // 3) * H))
display(grid)

## 4. The ground truth stays aligned
The camera warps the image (rotation + shear), but the same transform is applied to the
PAGE coordinates. Overlaying them confirms the baselines (green) and line polygons (red)
still sit exactly on the text after the warp.

In [ ]:
import re
from PIL import ImageDraw

def overlay(name):
    im = Image.open(f"data/synth/images/{name}.jpg").convert("RGB")
    xml = Path(f"data/synth/page/{name}.xml").read_text(encoding="utf-8")
    d = ImageDraw.Draw(im)
    for m in re.finditer(r'<Coords points="([^"]+)"', xml):
        pts = [tuple(map(int, p.split(","))) for p in m.group(1).split()]
        if len(pts) == 4:
            d.polygon(pts, outline=(230, 30, 30))
    for m in re.finditer(r'<Baseline points="([^"]+)"', xml):
        pts = [tuple(map(int, p.split(","))) for p in m.group(1).split()]
        d.line(pts, fill=(0, 200, 0), width=2)
    return im

display(overlay("synth_00000").resize((480, 360)))

## 5. The PAGE-XML
The label follows stone convention: all-caps Capitalis, **V** for U/V, **I** for I/J, no
abbreviation expansion, and a middot between words. This is exactly what a recognition
model should output.

In [ ]:
print(Path("data/synth/page/synth_00000.xml").read_text(encoding="utf-8"))

## 6. Next steps and options

`data/synth/` now holds a fresh training set. Run **`htr_cluster_demo.ipynb`** to train on it.

- **Scriptio continua:** add `--no-dots` to drop the middots.
- **More or fewer images:** change `--n`.
- **Your own texts:** replace `inscription_texts.txt` (one inscription per line, its lines joined by `|`).
- **More variety:** drop extra OFL fonts into `fonts/` or stone photos into `stone_images/` — no code change needed.

To compile and train directly:

```bash
ketos compile -f page -o synth.arrow data/synth/page/*.xml
ketos train -f binary -o model --device cuda:0 synth.arrow
```